In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.dummy import DummyRegressor
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')

print("✓ All imports successful")
print(f"  MLflow   : {mlflow.__version__}")
print(f"  XGBoost  : {xgb.__version__}")
print(f"  LightGBM : {lgb.__version__}")

C:\Users\Mansoor\anaconda3\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


✓ All imports successful
  MLflow   : 3.12.0
  XGBoost  : 2.0.3
  LightGBM : 4.3.0


In [17]:
import pandas as pd
import os

# Re-read using pyarrow engine explicitly with a fallback to fastparquet
# If that also fails, we convert all files to CSV

files = {
    "train" : "../data/processed/featured/train_featured.parquet",
    "val"   : "../data/processed/featured/val_featured.parquet",
    "test"  : "../data/processed/featured/test_featured.parquet",
}

for split, path in files.items():
    try:
        df = pd.read_parquet(path, engine="fastparquet")
        print(f"✓ {split} loaded via fastparquet — {df.shape}")
    except Exception as e1:
        print(f"✗ fastparquet failed for {split}: {e1}")
        try:
            df = pd.read_parquet(path, engine="pyarrow")
            print(f"✓ {split} loaded via pyarrow — {df.shape}")
        except Exception as e2:
            print(f"✗ pyarrow also failed for {split}: {e2}")

✓ train loaded via fastparquet — (67493, 70)
✓ val loaded via fastparquet — (14473, 70)
✓ test loaded via fastparquet — (14471, 70)


In [18]:
df_train = pd.read_parquet("../data/processed/featured/train_featured.parquet", engine="fastparquet")
df_val   = pd.read_parquet("../data/processed/featured/val_featured.parquet",   engine="fastparquet")
df_test  = pd.read_parquet("../data/processed/featured/test_featured.parquet",  engine="fastparquet")

with open("../data/processed/featured/feature_metadata.json") as f:
    feature_meta = json.load(f)

FEATURES = feature_meta["final_features"]
TARGET   = feature_meta["target"]

X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_val   = df_val[FEATURES]
y_val   = df_val[TARGET]

print("✓ Data loaded")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  Features: {len(FEATURES)}")
print(f"  Target  : {TARGET}")
print(f"\n  Target stats (train):")
print(f"    Mean   : {y_train.mean():.4f}")
print(f"    Std    : {y_train.std():.4f}")
print(f"    Min    : {y_train.min():.4f}")
print(f"    Max    : {y_train.max():.4f}")

✓ Data loaded
  X_train : (67493, 44)
  X_val   : (14473, 44)
  Features: 44
  Target  : lead_time_variance

  Target stats (train):
    Mean   : -11.2360
    Std    : 9.6564
    Min    : -83.0706
    Max    : 97.9718


In [19]:
def compute_metrics(y_true, y_pred, split="val"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    # MAPE — skip zeros to avoid division by zero
    nonzero = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100

    # Directional accuracy — correct late/early prediction
    dir_acc = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100

    return {
        f"{split}_mae"      : round(mae, 4),
        f"{split}_rmse"     : round(rmse, 4),
        f"{split}_r2"       : round(r2, 4),
        f"{split}_mape"     : round(mape, 4),
        f"{split}_dir_acc"  : round(dir_acc, 4),
    }

def print_metrics(name, metrics):
    print(f"\n  [{name}]")
    print(f"    MAE           : {metrics.get('val_mae', metrics.get('train_mae')):.4f} days")
    print(f"    RMSE          : {metrics.get('val_rmse', metrics.get('train_rmse')):.4f} days")
    print(f"    R²            : {metrics.get('val_r2', metrics.get('train_r2')):.4f}")
    print(f"    MAPE          : {metrics.get('val_mape', metrics.get('train_mape')):.2f}%")
    print(f"    Dir Accuracy  : {metrics.get('val_dir_acc', metrics.get('train_dir_acc')):.2f}%")

print("✓ Metrics helper defined")

✓ Metrics helper defined


In [20]:
EXPERIMENT_NAME = "procurement_ltv"

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)

exp = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"✓ MLflow experiment set")
print(f"  Name       : {EXPERIMENT_NAME}")
print(f"  Experiment : {exp.experiment_id}")
print(f"  Tracking   : ../mlruns")

✓ MLflow experiment set
  Name       : procurement_ltv
  Experiment : 615752846064675776
  Tracking   : ../mlruns


In [21]:
print("=" * 60)
print("  MODEL 0 — Naive Baseline (Predict Mean)")
print("=" * 60)

with mlflow.start_run(run_name="baseline_dummy"):
    model = DummyRegressor(strategy="mean")
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    mlflow.log_params({"strategy": "mean", "model_type": "DummyRegressor"})
    mlflow.log_metrics(all_metrics)

    print("\n  Train metrics:")
    print_metrics("Baseline Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("Baseline Val",   val_metrics)
    print(f"\n  Predicted constant : {model.constant_[0]:.4f} days")
    print(f"\n  ✓ Logged to MLflow")

  MODEL 0 — Naive Baseline (Predict Mean)

  Train metrics:

  [Baseline Train]
    MAE           : 6.7455 days
    RMSE          : 9.6563 days
    R²            : 0.0000
    MAPE          : 692.82%
    Dir Accuracy  : 91.93%

  Val metrics:

  [Baseline Val]
    MAE           : 6.8778 days
    RMSE          : 10.5116 days
    R²            : -0.0001
    MAPE          : 439.68%
    Dir Accuracy  : 91.78%


TypeError: unsupported format string passed to numpy.ndarray.__format__

In [22]:
print("=" * 60)
print("  MODEL 1 — Linear Regression")
print("=" * 60)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

with mlflow.start_run(run_name="linear_regression"):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LinearRegression())
    ])
    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)
    val_pred   = pipe.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    params = {"model_type": "LinearRegression", "scaler": "StandardScaler"}
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.sklearn.log_model(pipe, "model")

    print("\n  Train metrics:")
    print_metrics("Linear Regression Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("Linear Regression Val",   val_metrics)
    print(f"\n  ✓ Logged to MLflow")

  MODEL 1 — Linear Regression


2026/05/08 09:44:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 09:44:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Train metrics:

  [Linear Regression Train]
    MAE           : 5.0882 days
    RMSE          : 7.6008 days
    R²            : 0.3804
    MAPE          : 498.86%
    Dir Accuracy  : 91.85%

  Val metrics:

  [Linear Regression Val]
    MAE           : 5.3675 days
    RMSE          : 8.7367 days
    R²            : 0.3091
    MAPE          : 252.87%
    Dir Accuracy  : 91.25%

  ✓ Logged to MLflow


In [23]:
print("=" * 60)
print("  MODEL 2 — Ridge Regression")
print("=" * 60)

with mlflow.start_run(run_name="ridge_regression"):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0))
    ])
    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)
    val_pred   = pipe.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    params = {"model_type": "Ridge", "alpha": 1.0, "scaler": "StandardScaler"}
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.sklearn.log_model(pipe, "model")

    print("\n  Train metrics:")
    print_metrics("Ridge Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("Ridge Val",   val_metrics)
    print(f"\n  ✓ Logged to MLflow")

  MODEL 2 — Ridge Regression


2026/05/08 09:44:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 09:44:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Train metrics:

  [Ridge Train]
    MAE           : 5.0882 days
    RMSE          : 7.6008 days
    R²            : 0.3804
    MAPE          : 498.85%
    Dir Accuracy  : 91.85%

  Val metrics:

  [Ridge Val]
    MAE           : 5.3675 days
    RMSE          : 8.7367 days
    R²            : 0.3091
    MAPE          : 252.87%
    Dir Accuracy  : 91.25%

  ✓ Logged to MLflow


In [24]:
print("=" * 60)
print("  MODEL 3 — Lasso Regression")
print("=" * 60)

with mlflow.start_run(run_name="lasso_regression"):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Lasso(alpha=0.1, max_iter=5000))
    ])
    pipe.fit(X_train, y_train)

    train_pred = pipe.predict(X_train)
    val_pred   = pipe.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    # Log which features Lasso zeroed out
    lasso_coefs = pipe.named_steps["model"].coef_
    n_zero = (lasso_coefs == 0).sum()

    params = {
        "model_type": "Lasso", "alpha": 0.1,
        "max_iter": 5000, "scaler": "StandardScaler",
        "features_zeroed": int(n_zero)
    }
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.sklearn.log_model(pipe, "model")

    print("\n  Train metrics:")
    print_metrics("Lasso Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("Lasso Val",   val_metrics)
    print(f"\n  Features zeroed by Lasso : {n_zero} / {len(FEATURES)}")
    if n_zero > 0:
        zeroed = [FEATURES[i] for i, c in enumerate(lasso_coefs) if c == 0]
        print(f"  Zeroed features: {zeroed}")
    print(f"\n  ✓ Logged to MLflow")

  MODEL 3 — Lasso Regression


2026/05/08 09:44:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 09:44:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Train metrics:

  [Lasso Train]
    MAE           : 5.0838 days
    RMSE          : 7.6285 days
    R²            : 0.3759
    MAPE          : 507.29%
    Dir Accuracy  : 91.89%

  Val metrics:

  [Lasso Val]
    MAE           : 5.3561 days
    RMSE          : 8.7645 days
    R²            : 0.3047
    MAPE          : 253.49%
    Dir Accuracy  : 91.31%

  Features zeroed by Lasso : 21 / 44
  Zeroed features: ['purchase_dow', 'purchase_month', 'purchase_hour', 'purchase_is_weekend', 'purchase_near_holiday', 'total_price', 'avg_price', 'product_width_cm', 'product_volume_cm3', 'volumetric_weight', 'product_name_lenght', 'product_description_lenght', 'freight_per_kg', 'freight_price_ratio', 'is_heavy_product', 'total_payment_value', 'payment_installments', 'payment_methods_used', 'customer_order_count', 'is_repeat_customer', 'category_avg_variance']

  ✓ Logged to MLflow


In [25]:
print("=" * 60)
print("  MODEL 4 — Random Forest Regressor")
print("=" * 60)

with mlflow.start_run(run_name="random_forest"):
    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=10,
        min_samples_split=20,
        max_features=0.5,
        n_jobs=-1,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred   = model.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    params = {
        "model_type"       : "RandomForest",
        "n_estimators"     : 300,
        "max_depth"        : 12,
        "min_samples_leaf" : 10,
        "min_samples_split": 20,
        "max_features"     : 0.5,
        "random_state"     : 42,
    }
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.sklearn.log_model(model, "model")

    # Top 10 feature importances
    fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

    print("\n  Train metrics:")
    print_metrics("Random Forest Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("Random Forest Val",   val_metrics)
    print(f"\n  Top 10 Feature Importances:")
    for feat, imp in fi.head(10).items():
        bar = "█" * int(imp * 200)
        print(f"    {feat:<40} {imp:.4f}  {bar}")
    print(f"\n  ✓ Logged to MLflow")

  MODEL 4 — Random Forest Regressor


2026/05/08 09:45:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 09:45:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Train metrics:

  [Random Forest Train]
    MAE           : 4.1923 days
    RMSE          : 6.3932 days
    R²            : 0.5617
    MAPE          : 343.03%
    Dir Accuracy  : 92.76%

  Val metrics:

  [Random Forest Val]
    MAE           : 4.8751 days
    RMSE          : 8.2918 days
    R²            : 0.3777
    MAPE          : 207.28%
    Dir Accuracy  : 91.68%

  Top 10 Feature Importances:
    estimated_delivery_days                  0.5153  ███████████████████████████████████████████████████████████████████████████████████████████████████████
    seller_avg_variance                      0.0999  ███████████████████
    purchase_month                           0.0583  ███████████
    seller_late_rate                         0.0447  ████████
    customer_state_avg_variance              0.0326  ██████
    seller_std_variance                      0.0267  █████
    same_state                               0.0245  ████
    purchase_quarter                         0.0238  ████
   

In [26]:
print("=" * 60)
print("  MODEL 5 — XGBoost Regressor")
print("=" * 60)

with mlflow.start_run(run_name="xgboost"):
    model_xgb = xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
        eval_metric="rmse",
        early_stopping_rounds=30,
    )
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    train_pred = model_xgb.predict(X_train)
    val_pred   = model_xgb.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    params = {
        "model_type"         : "XGBoost",
        "n_estimators"       : 500,
        "best_iteration"     : model_xgb.best_iteration,
        "max_depth"          : 6,
        "learning_rate"      : 0.05,
        "subsample"          : 0.8,
        "colsample_bytree"   : 0.8,
        "min_child_weight"   : 10,
        "reg_alpha"          : 0.1,
        "reg_lambda"         : 1.0,
        "early_stopping"     : 30,
    }
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.xgboost.log_model(model_xgb, "model")

    fi_xgb = pd.Series(
        model_xgb.feature_importances_, index=FEATURES
    ).sort_values(ascending=False)

    print(f"\n  Best iteration (early stopping) : {model_xgb.best_iteration}")
    print("\n  Train metrics:")
    print_metrics("XGBoost Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("XGBoost Val",   val_metrics)
    print(f"\n  Top 10 Feature Importances:")
    for feat, imp in fi_xgb.head(10).items():
        bar = "█" * int(imp * 300)
        print(f"    {feat:<40} {imp:.4f}  {bar}")
    print(f"\n  ✓ Logged to MLflow")

  MODEL 5 — XGBoost Regressor


2026/05/08 09:46:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



  Best iteration (early stopping) : 493

  Train metrics:

  [XGBoost Train]
    MAE           : 3.9241 days
    RMSE          : 5.9342 days
    R²            : 0.6223
    MAPE          : 353.95%
    Dir Accuracy  : 93.07%

  Val metrics:

  [XGBoost Val]
    MAE           : 4.7010 days
    RMSE          : 8.1193 days
    R²            : 0.4033
    MAPE          : 197.31%
    Dir Accuracy  : 91.52%

  Top 10 Feature Importances:
    estimated_delivery_days                  0.1577  ███████████████████████████████████████████████
    same_state                               0.0901  ███████████████████████████
    purchase_year                            0.0595  █████████████████
    purchase_quarter                         0.0574  █████████████████
    seller_avg_variance                      0.0499  ██████████████
    purchase_month                           0.0410  ████████████
    unique_sellers                           0.0407  ████████████
    cross_region                          

In [27]:
print("=" * 60)
print("  MODEL 6 — LightGBM Regressor")
print("=" * 60)

with mlflow.start_run(run_name="lightgbm"):
    model_lgb = lgb.LGBMRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False),
                   lgb.log_evaluation(period=-1)]
    )

    train_pred = model_lgb.predict(X_train)
    val_pred   = model_lgb.predict(X_val)

    train_metrics = compute_metrics(y_train, train_pred, "train")
    val_metrics   = compute_metrics(y_val,   val_pred,   "val")
    all_metrics   = {**train_metrics, **val_metrics}

    params = {
        "model_type"        : "LightGBM",
        "n_estimators"      : 500,
        "best_iteration"    : model_lgb.best_iteration_,
        "max_depth"         : 6,
        "learning_rate"     : 0.05,
        "num_leaves"        : 63,
        "subsample"         : 0.8,
        "colsample_bytree"  : 0.8,
        "min_child_samples" : 20,
        "reg_alpha"         : 0.1,
        "reg_lambda"        : 1.0,
        "early_stopping"    : 30,
    }
    mlflow.log_params(params)
    mlflow.log_metrics(all_metrics)
    mlflow.lightgbm.log_model(model_lgb, "model")

    fi_lgb = pd.Series(
        model_lgb.feature_importances_, index=FEATURES
    ).sort_values(ascending=False)

    print(f"\n  Best iteration (early stopping) : {model_lgb.best_iteration_}")
    print("\n  Train metrics:")
    print_metrics("LightGBM Train", train_metrics)
    print("\n  Val metrics:")
    print_metrics("LightGBM Val",   val_metrics)
    print(f"\n  Top 10 Feature Importances:")
    for feat, imp in fi_lgb.head(10).items():
        bar = "█" * int(imp / 200)
        print(f"    {feat:<40} {imp:.0f}  {bar}")
    print(f"\n  ✓ Logged to MLflow")

  MODEL 6 — LightGBM Regressor


2026/05/08 09:46:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 09:46:45 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  Best iteration (early stopping) : 500

  Train metrics:

  [LightGBM Train]
    MAE           : 3.9761 days
    RMSE          : 6.0437 days
    R²            : 0.6083
    MAPE          : 326.08%
    Dir Accuracy  : 93.01%

  Val metrics:

  [LightGBM Val]
    MAE           : 4.7091 days
    RMSE          : 8.1174 days
    R²            : 0.4036
    MAPE          : 200.98%
    Dir Accuracy  : 91.48%

  Top 10 Feature Importances:
    seller_std_variance                      2029  ██████████
    estimated_delivery_days                  1993  █████████
    purchase_month                           1928  █████████
    customer_state_avg_variance              1560  ███████
    seller_avg_variance                      1400  ███████
    seller_late_rate                         1240  ██████
    total_freight                            1074  █████
    product_description_lenght               811  ████
    seller_order_count                       779  ███
    freight_price_ratio               

In [29]:
print("\n")
print("=" * 85)
print("  MODEL COMPARISON SUMMARY — VALIDATION SET")
print("=" * 85)
print(f"\n  {'Model':<25} {'MAE':>8} {'RMSE':>8} {'R²':>8} {'MAPE%':>8} {'DirAcc':>9}")
print("  " + "-" * 73)

client = mlflow.tracking.MlflowClient(tracking_uri="../mlruns")
exp    = client.get_experiment_by_name(EXPERIMENT_NAME)
runs   = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.val_mae ASC"]
)

# Build map by run name — handle duplicates by keeping best MAE
run_map = {}
for r in runs:
    name = r.data.tags.get("mlflow.runName", "")
    mae  = r.data.metrics.get("val_mae", float("inf"))
    if name not in run_map or mae < run_map[name].data.metrics.get("val_mae", float("inf")):
        run_map[name] = r

# Print in fixed order
display_order = [
    ("Baseline (Mean)",   "baseline_dummy"),
    ("Linear Regression", "linear_regression"),
    ("Ridge",             "ridge_regression"),
    ("Lasso",             "lasso_regression"),
    ("Random Forest",     "random_forest"),
    ("XGBoost",           "xgboost"),
    ("LightGBM",          "lightgbm"),
]

for model_name, run_name in display_order:
    if run_name in run_map:
        m = run_map[run_name].data.metrics
        mae     = m.get("val_mae",     0)
        rmse    = m.get("val_rmse",    0)
        r2      = m.get("val_r2",      0)
        mape    = m.get("val_mape",    0)
        dir_acc = m.get("val_dir_acc", 0)
        print(f"  {model_name:<25} {mae:>8.4f} {rmse:>8.4f} {r2:>8.4f} "
              f"{mape:>8.2f} {dir_acc:>8.2f}%")
    else:
        # Fallback — print all available run names for debugging
        print(f"  {model_name:<25} {'NOT FOUND':>50}")

print(f"\n  Available runs in MLflow:")
for name, r in run_map.items():
    m = r.data.metrics
    print(f"    '{name}' → val_mae={m.get('val_mae','?')}")

print(f"\n  Primary metric : MAE (lower is better)")
print(f"  Proceeding to tuning : XGBoost + LightGBM")
print("=" * 85)



  MODEL COMPARISON SUMMARY — VALIDATION SET

  Model                          MAE     RMSE       R²    MAPE%    DirAcc
  -------------------------------------------------------------------------
  Baseline (Mean)             6.8778  10.5116  -0.0001   439.68    91.78%
  Linear Regression           5.3675   8.7367   0.3091   252.87    91.25%
  Ridge                       5.3675   8.7367   0.3091   252.87    91.25%
  Lasso                       5.3561   8.7645   0.3047   253.49    91.31%
  Random Forest               4.8751   8.2918   0.3777   207.28    91.68%
  XGBoost                     4.7010   8.1193   0.4033   197.31    91.52%
  LightGBM                    4.7091   8.1174   0.4036   200.98    91.48%

  Available runs in MLflow:
    'xgboost' → val_mae=4.701
    'lightgbm' → val_mae=4.7091
    'random_forest' → val_mae=4.8751
    'lasso_regression' → val_mae=5.3561
    'ridge_regression' → val_mae=5.3675
    'linear_regression' → val_mae=5.3675
    'baseline_dummy' → val_mae=6.877